In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

In [ ]:
# --- Connection info ---
PGHOST = "136.112.214.240"
PGPORT = "5432"
PGDATABASE = "feature-service-db"
PGUSER = "user"
PGPASSWORD = "postgres"

# --- Create SQLAlchemy engine ---
engine = create_engine(f"postgresql+psycopg2://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}")

# --- Test connection ---
with engine.begin() as conn:
    result = conn.execute(text("SELECT version();"))
    print("Connected to:", result.scalar())

In [ ]:
# Define Query
start_date = "2015-1-01 00:00:00"
end_date = "2016-12-31 23:59:00"

query = f"""
SELECT *
FROM x_min_price_data
WHERE datetime BETWEEN '{start_date}' AND '{end_date}'
ORDER BY datetime;
"""

df = pd.read_sql(query, engine)
df.head()

In [ ]:
df.head(60)

In [ ]:
# Drop 'price_vol_weight_avg' column
df = df.drop(columns=['price_vol_weight_avg'])

# Ensure datetime is in datetime format
df['datetime'] = pd.to_datetime(df['datetime'])

# Sort for consistency
df = df.sort_values(['ticker', 'datetime'])

# Define aggregation rules
agg_funcs = {
        'price_open': 'first',
        'price_high': 'max',
        'price_low': 'min',
        'price_close': 'last',
        'price_vol': 'sum'
    }

# Group by ticker and resample
df_hour = (
    df.set_index('datetime')
          .groupby('ticker')
          .resample('1h')
          .agg(agg_funcs)
          .dropna()
          .reset_index()
    )

# Reorder columns
df_hour = df_hour[['datetime', 'ticker', 'price_open', 'price_close', 'price_high', 'price_low', 'price_vol']]

df_hour

In [ ]:
# Bulk Insert to SQL Table
df_hour.to_sql(
    name='x_hour_price_data',
    con=engine,
    if_exists='append',   # options: 'fail', 'replace', 'append'
    index=False,          # don't include DataFrame index
    chunksize=1000,       # send in batches of 1000 for performance
    method='multi'        # enables bulk insert
)
